# Notebook 02: Exploratory Data Analysis (EDA)
## SkillGraph AI - Labor Market & Skill Distribution Insights

This notebook explores job title frequency, geographic distribution, salary bands, and skill co-occurrence patterns across real postings.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

conn = sqlite3.connect("../data/skillgraph.db")

### 1. Top In-Demand Occupations & Job Titles

In [ ]:
top_titles = pd.read_sql_query("""
    SELECT title, COUNT(*) as posting_count
    FROM job_postings
    GROUP BY title
    ORDER BY posting_count DESC
    LIMIT 15
""", conn)

plt.figure(figsize=(10, 5))
sns.barplot(data=top_titles, x='posting_count', y='title', palette='viridis')
plt.title("Top 15 Most Frequent Job Titles")
plt.xlabel("Posting Count")
plt.ylabel("Job Title")
plt.tight_layout()
plt.show()

### 2. Geographic Distribution & Remote Work Opportunities

In [ ]:
loc_df = pd.read_sql_query("""
    SELECT location, COUNT(*) as count
    FROM job_postings
    WHERE location IS NOT NULL AND location != ''
    GROUP BY location
    ORDER BY count DESC
    LIMIT 12
""", conn)

plt.figure(figsize=(10, 4))
sns.barplot(data=loc_df, x='count', y='location', color='#0284c7')
plt.title("Top Job Posting Geographic Hubs")
plt.xlabel("Number of Postings")
plt.ylabel("Location")
plt.tight_layout()
plt.show()

### 3. Top Canonical Skills by Demand Share

In [ ]:
top_skills = pd.read_sql_query("""
    SELECT s.canonical_name, s.category, SUM(o.posting_count) as total_mentions, AVG(o.demand_share) as avg_share
    FROM skill_demand_observations o
    JOIN skills s ON o.skill_id = s.id
    GROUP BY s.id, s.canonical_name, s.category
    ORDER BY total_mentions DESC
    LIMIT 20
""", conn)

display(top_skills.head(10))

plt.figure(figsize=(10, 6))
sns.barplot(data=top_skills, x='total_mentions', y='canonical_name', hue='category', dodge=False)
plt.title("Top 20 Canonical Skills in Demand")
plt.xlabel("Total Observation Count")
plt.ylabel("Skill")
plt.tight_layout()
plt.show()
conn.close()